# 08 - Post-hoc calibration correction

**Audience:** re-applying a saved EyeTrackingProfile to a recording.

When to use:
- A better calibration profile became available after the recording was made.
- You want to compare the same recording under different profiles.
- The session was recorded before live calibration was standard practice.

When NOT to use: do not apply correction to a CSV that already had a
profile active live (read `ctx.metadata['Profile']` to check). The two
corrections compound and you'll over-shoot.

## What you'll get

- Profile inspection (offsets, gains, per-eye identity check).
- Apply correction; new CSV written next to the input.
- Per-sample angular shift histogram.
- Re-run quality + classifier on corrected data and diff key metrics.

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
print(ctx)

## 1. Profile inspection

In [ ]:
if ctx.profile_path is None:
    print(
        "No profile JSON found next to the CSV.\n\n"
        "This notebook needs an `<HMD>_<timestamp>.json` produced by the calibrator. "
        "Drop one alongside your recording, or set ctx.profile_path = Path('/path/to/profile.json') "
        "and re-run from this cell. The bundled sample doesn't ship a profile, "
        "so this notebook will short-circuit on a fresh checkout."
    )
    profile = None
else:
    profile = ela.load_profile(ctx.profile_path)
    print(f"Profile name:    {profile.profile_name}")
    print(f"Schema version:  {profile.schema_version}")
    cg = profile.combined_gaze
    print(f"\nCombined correction:")
    print(f"  yaw offset:    {cg.gaze_yaw_offset_deg:+.4f} deg")
    print(f"  pitch offset:  {cg.gaze_pitch_offset_deg:+.4f} deg")
    print(f"  yaw gain:      {cg.gaze_yaw_gain:.4f}")
    print(f"  pitch gain:    {cg.gaze_pitch_gain:.4f}")
    print(f"\nPer-eye identity? left={profile.left_eye.is_identity()}  right={profile.right_eye.is_identity()}")

## 2. Sanity check: was a profile already applied at recording time?

If `Profile` metadata is anything other than `none`, the live recording
ALREADY had a profile baked in. Applying a second correction will
compound. Stop and reconsider.

In [ ]:
live_profile = ctx.metadata.get("Profile", "(unset)")
print(f"Live Profile metadata: {live_profile}")
if live_profile and live_profile != "none":
    print("\nWARNING: a profile was active during recording. "
          "Applying another correction now will compound the offsets.")

## 3. Apply correction

In [ ]:
if profile is None:
    print("Skipping apply step (no profile loaded).")
    out_path = None
    corr_data = None
else:
    out_path = ctx.csv_path.with_name(ctx.csv_path.stem + "_corrected.csv")
    stats = ela.apply_profile_to_csv(ctx.profile_path, ctx.csv_path, out_path, overwrite=True)
    print(f"Wrote {out_path.name}")
    for k, v in stats.items():
        print(f"  {k:<32} {v}")
    corr_data = ela.load_eyetracking(out_path)

## 4. Per-sample angular shift

In [ ]:
if corr_data is None:
    print("Skipping angular-shift visualization (no corrected CSV).")
else:
    raw = ctx.data.df[["combined_dir_x", "combined_dir_y", "combined_dir_z"]].to_numpy(dtype=float)
    cor = corr_data.df[["combined_dir_x", "combined_dir_y", "combined_dir_z"]].to_numpy(dtype=float)
    valid = np.all(np.isfinite(raw) & np.isfinite(cor), axis=1)
    raw_n = raw[valid] / np.linalg.norm(raw[valid], axis=1, keepdims=True)
    cor_n = cor[valid] / np.linalg.norm(cor[valid], axis=1, keepdims=True)
    dots = np.clip(np.einsum("ij,ij->i", raw_n, cor_n), -1, 1)
    delta = np.degrees(np.arccos(dots))
    print(f"Angular shift  (n={len(delta)}):")
    print(f"  median: {np.median(delta):.3f} deg")
    print(f"  p95:    {np.percentile(delta, 95):.3f} deg")
    print(f"  max:    {np.max(delta):.3f} deg")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.hist(delta, bins=40)
    ax.set_xlabel("per-sample angular shift (deg)")
    ax.set_ylabel("count")
    ax.set_title("Distribution of correction magnitude")
    plt.tight_layout()
    plt.show()

## 5. Before/after metrics diff

In [ ]:
def quick_metrics(d):
    df = d.df
    ts = d.get_timestamps()
    out = {}
    if all(c in df.columns for c in ("combined_dir_x", "combined_dir_y", "combined_dir_z")):
        dx = df["combined_dir_x"].to_numpy(dtype=float)
        dy = df["combined_dir_y"].to_numpy(dtype=float)
        dz = df["combined_dir_z"].to_numpy(dtype=float)
        yaw = np.degrees(np.arctan2(dx, dz))
        pitch = -np.degrees(np.arcsin(np.clip(dy, -1, 1)))
        mv = ela.detect_eye_movements(yaw, pitch, ts)
        out["n_fixations"] = len(mv["fixations"])
        out["n_saccades"] = len(mv["saccades"])
        if mv["fixations"]:
            out["median_fix_ms"] = float(np.median([f.duration*1000 for f in mv["fixations"]]))
    return out

if corr_data is None:
    print("Skipping before/after diff (no corrected CSV).")
else:
    before = quick_metrics(ctx.data)
    after = quick_metrics(corr_data)
    diff = pd.DataFrame({"before": before, "after": after})
    diff["delta"] = diff["after"] - diff["before"]
    diff